## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# 04b - Estimator validation on real RCTs + refutations on the observational effects
(a) Hillstrom (and Criteo if available): model ATE vs experimental diff-in-means, Qini, heterogeneity calibration.
(b) Placebo-treatment, random-common-cause, subset refuters and E-values for every clause effect on Lending Club.

In [ ]:
import numpy as np, pandas as pd
from causal_rag import rct_validation as rv
rows = []
try:
    h = rv.load_hillstrom(); L, T, y = rv.hillstrom_layers(h)
    if SMOKE: idx = np.random.default_rng(0).choice(len(y), 15000, replace=False); L=[x[idx] for x in L]; T=T[idx]; y=y[idx]
    out, last = rv.evaluate_rct(L, T, y, known_e=float(T.mean()), epochs=EPOCHS)
    out.insert(0, 'dataset', 'Hillstrom'); display(out); display(last[1]); rows.append(out)
except Exception as e:
    print('Hillstrom RCT validation SKIPPED (does not affect later stages):', type(e).__name__, str(e)[:400])
if not SMOKE:
    try:
        c = rv.load_criteo(); feats = [k for k in c.columns if k.startswith('f')]
        X = ((c[feats]-c[feats].mean())/c[feats].std()).values.astype('float32')
        o2, _ = rv.evaluate_rct([X], c.treatment.values.astype('float32'), c.visit.values.astype('float32'),
                                known_e=float(c.treatment.mean()), epochs=EPOCHS, flat_only=True)
        o2.insert(0, 'dataset', 'Criteo'); display(o2); rows.append(o2)
    except Exception as e: print('Criteo skipped:', type(e).__name__, e)
if rows: pd.concat(rows).to_csv(paths.art('rct_results.csv'), index=False)

In [ ]:
import json
from causal_rag import refute
from causal_rag.config import TREATMENTS, Featurizer, fit_medians, add_features
df = pd.read_parquet(paths.art('lc_clean.parquet'))
tr = (df.role=='train').values; te = (df.role=='test').values
d = add_features(df, fit_medians(df, tr)); yy = d.default.values
outs = []
for tk in TREATMENTS:
    fz = Featurizer(tk).fit(d, tr); X = np.hstack(fz.transform(d)); Tk = fz.treatment(d)
    outs.append(refute.run_refutations(X[te], Tk[te], yy[te], n_rep=1 if SMOKE else 3, max_n=8000 if SMOKE else 60000, name=tk))
R = pd.concat(outs); R.to_csv(paths.art('refutations.csv'), index=False)
display(refute.pass_rates(R)); display(R[R.test.isin(['estimate','e_value_point','e_value_ci_bound'])])